## Tool Calling Implementation
### Simple Calculator

In [2]:
!pip install anthropic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 17.6 MB/s eta 0:00:00


In [10]:
import anthropic
from google.colab import userdata

In [11]:
client = anthropic.Anthropic(api_key=userdata.get('dinkey'))

In [12]:
# Precise description of the tool that claude needs to know

tools1 = [
    {
        "name": "calculate",
        "description": "performs simple math. Use this whenever user asks you to calculate something.",
        "input_schema": {
            "type": "object", # arguments will be a dictionary
            "properties": { # all possible arguments
                "expression": { # argument name
                    "type": "string", # what data type it should be
                    "description": "The math expression to evaluate, eg. '2 + 2' or '10 * 20.'" # claude reads this to know what to put here.
                }
            },
            "required": ["expression"] # [necessary] idk, have to check later
        }
    }
]
# what are the keys in input_schema?
# why is it an array but theres only one json stored in it?

In [13]:
def calculate(expression):
  try:
    result = eval(expression)
    return str(result)
  except Exception as e:
    return str(e)

In [18]:
conversation = []

def chat(user_message):
  conversation.append({"role": "user", "content": user_message})

  response = client.messages.create(
      model="claude-haiku-4-5-20251001",
      max_tokens=256,
      tools=tools1,
      messages=conversation
  )

  # handle tool cal
  if response.stop_reason == "tool_use":
    tool_use = response.content[0]
    tool_name = tool_use.name
    tool_input = tool_use.input
    result = calculate(tool_input)
    print(f"claude wants to call {tool_name}")
    print(f"with arguments {tool_input}")

    conversation.append({"role": "assistant", "content": response.content})

    # tells claude the result
    conversation.append({"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": tool_use.id, "content": result}
    ]})

    response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            tools=tools1,
            messages=conversation
        )

    reply = response.content[0].text
    conversation.append({"role": "assistant", "content": reply})
    print(f"Claude: {reply}\n")

  # if your prompt requires no tool calling
  else:
      reply = response.content[0].text
      conversation.append({"role": "assistant", "content": reply})
      print(f"Claude: {reply}\n")
chat("what is 2569 * 213?")
chat("multiply that by 3")
chat("what is the capital of France?")


claude wants to call calculate
with arguments {'expression': '2569 * 213'}
Claude: Let me try that again:

claude wants to call calculate
with arguments {'expression': '547197 * 3'}
Claude: I apologize for the technical difficulty. Let me calculate this directly for you:

2569 × 213 = 547,197

547,197 × 3 = **1,641,591**

Claude: The capital of France is **Paris**.

Paris is located in north-central France along the Seine River and is the country's largest city. It's known as the "City of Light" (Ville Lumière) and is famous for landmarks like the Eiffel Tower, Notre-Dame Cathedral, the Louvre Museum, and many other iconic sites.

